In [ ]:
### 합성곱 신경망을 활용한 이미지 분류

## 패션 MNIST 호출 및 훈련/검증 세트 추출

import keras
from sklearn.model_selection import train_test_split

(train_input, train_target), (test_input, test_target) = keras.datasets.fashion_mnist.load_data()

train_scaled = train_input.reshape(-1, 28, 28, 1) / 255.0 # 각 훈련 샘플을 3차원(높이, 너비, 채널) 변환 + 표준화

train_scaled, val_scaled, train_target, val_target = train_test_split(train_scaled, train_target, test_size=0.2, random_state=42)



## 합성곱 신경망 모델 제작

model = keras.Sequential()

model.add(keras.layers.Input(shape = (28, 28, 1))) # 입력층

model.add(keras.layers.Conv2D(32, kernel_size = 3, activation = 'relu', padding = 'same')) # 첫 번째 합성곱층
model.add(keras.layers.MaxPooling2D(2)) # 첫 번째 풀링

model.add(keras.layers.Conv2D(64, kernel_size = 3, activation = 'relu', padding = 'same')) # 두 번째 합성곱층
model.add(keras.layers.MaxPooling2D(2)) # 두 번째 풀링

model.add(keras.layers.Flatten()) # Flatten - 입력 1차원 펼치기
model.add(keras.layers.Dense(100, activation = 'relu')) # 은닉층
model.add(keras.layers.Dropout(0.4)) # 드롭아웃층
model.add(keras.layers.Dense(10, activation = 'softmax')) # 출력층

model.summary()



## 모델 컴파일 및 훈련 진행

model.compile(optimizer = 'adam', loss = 'sparse_categorical_crossentropy', metrics = ['accuracy'])

checkpoint_cb = keras.callbacks.ModelCheckpoint('best-cnn-model.keras', save_best_only = True)

early_stopping_cb = keras.callbacks.EarlyStopping(patience = 2, restore_best_weights = True)

import tensorflow as tf
print("TensorFlow version:", tf.__version__)
print("All devices:", tf.config.list_physical_devices())
print("GPU devices:", tf.config.list_physical_devices('GPU'))
print("train_scaled dtype:", train_scaled.dtype)

!nvidia-smi

history = model.fit(
    train_scaled, train_target,
    epochs=1,
    batch_size=128,
    validation_data=(val_scaled, val_target),
    verbose=1
)

history = model.fit(train_scaled, train_target, epochs= 20, validation_data=(val_scaled, val_target), callbacks = [checkpoint_cb, early_stopping_cb])

import matplotlib.pyplot as plt

plt.plot(history.history['loss'], label = 'train')
plt.plot(history.history['val_loss'], label = 'val')
plt.xlabel('epoch')
plt.ylabel('loss')
plt.legend()
plt.show()

model.evaluate(val_scaled, val_target) # 최적 시점 조기 종료 후 검증 세트 정확도 평가



## 검증 세트 샘플 1개 고찰

plt.imshow(val_scaled[0].reshape(28, 28), cmap = 'gray_r')
plt.show() # 검증 세트 0번째 입력 시각화

preds = model.predict(val_scaled[0].reshape(-1, 28, 28, 1)) # 검증 세트 0번째 입력 예측
# preds => 10개 카테고리에 대한 확률값 저장
print(preds)


plt.bar(range(1, 11), preds[0])
plt.xlabel('class')
plt.ylabel('probability')
plt.show() # 검증 세트 0번째 입력 예측 결과 그래프화

classes = ['티셔츠', '바지', '스웨터', '드레스', '코트', '샌달', '셔츠', '스니커즈', '가방', '앵클 부츠']

import numpy as np

print(classes[np.argmax(preds)]) # => preds 예측에서 가장 높은 값 인덱스 => classes에 적용(어떤 카테고리인지 확인)



# 최종 테스트 성능 평가
test_scaled = test_input.reshape(-1, 28, 28, 1) / 255.0

model.evaluate(test_scaled, test_target)
